# Leren Excel 라벨 기반 정밀도 EDA 및 키워드 실험

이 노트북은 `라벨링.xlsx`를 기준으로 문제 생성용 자료와 AI 튜터 검색용 자료를 분리해 분석합니다.

- 문제 생성: `확정 골드 = 2`를 양성으로 사용하여 Precision을 우선 확인합니다.
- 튜터 검색: `확정 골드 = 1 또는 2`를 양성으로 사용하여 Recall을 함께 확인합니다.
- 키워드 매칭은 최종 판정이 아니라 후보 추출 단계입니다.
- 이 노트북은 LLM API를 호출하지 않으며, 독립 검증 전의 현재 골드를 참고 기준으로만 사용합니다.

In [ ]:
from pathlib import Path
import json
import sys

# 노트북을 저장소 루트·notebook 디렉터리 중 어디에서 실행해도 동작하게 합니다.
ROOT = Path.cwd()
while not (ROOT / 'app').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.rebuild_precision_pipeline import run

# 기존 재사용 모듈을 호출합니다. Excel 원본은 읽기만 하고 수정하지 않습니다.
summary = run()
print('입력 파일:', summary['input_file'])
print('전체 행 수:', summary['row_count'])
print('LLM 호출 수:', summary['llm_calls'])
print('검수 큐:', summary['review_queue_count'], '건')

## 1. 데이터 규모·토큰·라벨 분포

아래 결과는 실행 시점에 Excel에서 다시 계산됩니다. 요청문에 적힌 참고 수치를 코드에 하드코딩하지 않습니다.

In [ ]:
token_stats = summary['text_eda']['token_count']
char_stats = summary['text_eda']['character_count']
print(f"기관별 건수: {summary['source_distribution']}")
print(f"설명 문자 수: 평균 {char_stats['average']:.1f}, 중앙값 {char_stats['median']}, 최대 {char_stats['max']}")
print(f"설명 토큰 수: 평균 {token_stats['average']:.1f}, 중앙값 {token_stats['median']}, 총 {token_stats['total']}")
print(f"토크나이저: {token_stats['tokenizer']} / {token_stats['encoding']}")
print(f"확정 골드 분포: {summary['gold_distribution']}")
print(f"평가자 A·B 일치율: {summary['label_quality']['reviewer_agreement_rate']:.1%}")
print(f"평가자 불일치: {summary['label_quality']['reviewer_disagreement_count']}건")
print(f"자료 품질 검토필요: {summary['label_quality']['quality_review_count']}건")

## 2. 기본 키워드와 확장 키워드 비교

기본 키워드는 핵심어만 사용하고, 확장 키워드는 약어·연관어·반대 개념까지 포함합니다. 두 전략을 문제 생성과 튜터 검색 목적별로 따로 평가합니다.

In [ ]:
def show_metric(strategy, usage, label):
    item = summary['keyword_evaluation'][strategy][usage]
    metric = item['metrics']
    print(
        f"{label:12} | 후보 {item['candidate_count']:3}건 | "
        f"Precision {metric['precision']:.1%} | Recall {metric['recall']:.1%} | "
        f"F1 {metric['f1']:.1%} | 혼동행렬 {item['matrix']}"
    )

for strategy, label in [('basic_core', '기본 키워드'), ('expanded', '확장 키워드')]:
    show_metric(strategy, 'question_generation', label + ' / 문제 생성')
    show_metric(strategy, 'tutor_retrieval', label + ' / 튜터 검색')

print('주의: 위 수치는 독립 골드가 아닌 현재 Excel 골드 기준선입니다.')

In [ ]:
import matplotlib.pyplot as plt

# 발표용으로 핵심 분포와 전략별 Precision·Recall을 한 번에 확인합니다.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

gold = summary['gold_distribution']
axes[0].bar(list(gold.keys()), list(gold.values()), color=['#94a3b8', '#fbbf24', '#2563eb'])
axes[0].set_title('확정 골드 관련도 분포')
axes[0].set_xlabel('관련도 점수')
axes[0].set_ylabel('건수')

labels = ['기본/문제', '확장/문제', '기본/튜터', '확장/튜터']
items = [
    summary['keyword_evaluation']['basic_core']['question_generation'],
    summary['keyword_evaluation']['expanded']['question_generation'],
    summary['keyword_evaluation']['basic_core']['tutor_retrieval'],
    summary['keyword_evaluation']['expanded']['tutor_retrieval'],
]
precision = [item['metrics']['precision'] for item in items]
recall = [item['metrics']['recall'] for item in items]
x = range(len(labels))
axes[1].bar([i - 0.18 for i in x], precision, width=0.36, label='Precision', color='#2563eb')
axes[1].bar([i + 0.18 for i in x], recall, width=0.36, label='Recall', color='#f59e0b')
axes[1].set_xticks(list(x), labels, rotation=20)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('전략·용도별 Precision / Recall')
axes[1].legend()

quality = summary['label_quality']
axes[2].bar(['A·B 불일치', '품질 검토', '검수 큐'], [quality['reviewer_disagreement_count'], quality['quality_review_count'], summary['review_queue_count']], color=['#dc2626', '#f97316', '#64748b'])
axes[2].set_title('검수 필요 규모')
axes[2].set_ylabel('건수')

plt.tight_layout()
plt.show()

## 3. 실험 해석 및 다음 단계

현재 결과에서는 확장 키워드가 문제 생성 후보의 재현율을 높였지만, 튜터 검색 목적에는 여전히 재현율이 낮습니다. 따라서 문제 생성은 키워드 후보 뒤에 LLM 정밀 분류와 Human Review를 두고, 튜터 검색은 키워드로 과도하게 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용하는 방향이 적절합니다.

평가자 A 라벨이 확정 골드와 동일하므로 현재 수치는 모델의 독립 성능이 아닙니다. 다음 실험에서는 평가자 A·B 불일치 10건, 자료 품질 검토필요 7건, 직접 관련 양성 표본을 포함한 독립 골드 세트를 확정한 뒤 같은 Notebook에서 임계값별 Precision·Recall 변화를 비교해야 합니다.